# Capstone build --- Chapter 1: What Is an Agent? The Governed Loop

This notebook opens a series that runs alongside the book. Across Chapters 1 through 16 it assembles the capstone banking complaint agent one layer at a time, so that the final chapter arrives at exactly the object that `agentlab.capstone.build_complaint_harness` returns. Each notebook adds the one layer its chapter introduces and carries the result forward.

The series follows a single principle: **build the governance scaffolding, import the trained artifacts.** The Qwen classifier head, the LoRA drafter and the calibrated GMS store are not retrained here; they are imported as they ship. What is constructed by hand is the architecture that surrounds them --- the loop, the reasoning trace, the typed action space, the gate stack, the plan, the memory and the escalation path.

Chapter 1 lays two of those layers, because it both draws the model/agent distinction and fixes the minimal agent loop: the classifier is wrapped as a typed action, then that action is driven by the real `run_loop`.

## The model is a component, not the agent

This chapter draws the distinction between a language model and an agent. A model maps a prompt to a completion; an agent is the governed process that decides when a model is called, checks what it returns and records that it happened. The first brick of the capstone is therefore not the agent but the thing the agent will wrap: the complaint classifier. Calling it directly shows what a raw model call is.

In [ ]:
from forgeloop.agents.models.complaint_classifier import get_default_classifier

clf = get_default_classifier()
label, confidence = clf.classify(
    'I was charged a $35 overdraft fee I did not authorize.'
)
print('raw model output:', label, round(confidence, 3))

The call returns an answer and nothing else. There is no schema constraining the shape of the output, no record that the call occurred, no bound on its cost and no check that classifying was the appropriate action at this point. A regulated decision cannot rest on a call with none of that around it. The agent is precisely what supplies it.

## Wrapping the model as a typed action

The capstone never calls the classifier in the way above. It wraps the same model behind a `Tool`: a named, typed action that declares an input schema and an output schema. The classifier becomes the tool's implementation, and the agent will only ever reach it by proposing a typed call that a harness runs on its behalf.

In [ ]:
from forgeloop.agents.capstone.banking_tools import classify_complaint

print('tool name   :', classify_complaint.name)
print('input       :', classify_complaint.input_schema.__name__)
print('output      :', classify_complaint.output_schema.__name__)
print('risk        :', classify_complaint.risk.value)
print('implemented :', classify_complaint.fn is not None)

The same model inference now sits behind a contract. The input schema names the fields a caller must supply, the output schema names what the tool guarantees to return, and the risk level records how much scrutiny a call to it warrants. This is the unit the agent operates on: not a model, but a typed action whose implementation happens to be a model.

## Placing the action in the minimal loop

The typed action above does not act on its own; something has to decide to call it, run it and read what comes back. That something is the agent loop. The rest of the chapter drives the tool with the real `agentlab.core.loop.run_loop`: an agent proposes an action, an environment executes it, the loop records the transition, and the cycle repeats until the agent finishes.

### The action space: one tool in a registry

The loop draws the actions it may propose from a registry. At this stage the registry holds the single classify tool wrapped above; Chapter~5 widens it to the full set. Registration is the reader's scaffolding; the tool itself is the imported artifact.

In [ ]:
from forgeloop.agents.tools import ToolRegistry

registry = ToolRegistry()
registry.register(classify_complaint)
print('registered:', [t.name for t in registry.all()])

### The environment: run a validated call

The environment is the boundary at which a proposed action becomes an effect. It validates the call's arguments against the tool's input schema, then runs the tool's implementation. There are no governance gates here yet --- this is the bare execution surface, and Chapter~6 is where the gate stack is inserted between the proposal and this call.

In [ ]:
class ToolEnvironment:
    """Minimal environment: validate a tool call, then run it."""

    def __init__(self, registry):
        self.registry = registry

    def step(self, action):
        parsed = self.registry.validate(action.tool_name, action.arguments)
        tool = self.registry.get(action.tool_name)
        output = tool.fn(**parsed.model_dump())
        return {'output': output}

### The agent: propose, then finish

An agent implements two methods: `propose_action`, which reads the current state and returns the next typed action, and `update`, which folds an observation back into the state. The minimal policy here proposes a classify call while no result has been recorded, and finishes once one has. The loop itself handles the `Finish` action; the agent only has to decide when to emit it.

In [ ]:
from forgeloop.agents.core.agent import BaseAgent
from forgeloop.agents.core.action import ToolCall, Finish

class ClassifyThenFinish(BaseAgent):
    def propose_action(self, state):
        if not state.tool_results:
            return ToolCall(
                tool_name='classify_complaint',
                arguments={'message': state.task.inputs['message']},
            )
        return Finish(output=state.tool_results[-1])

    def update(self, state, action, observation):
        new_state = state.model_copy(deep=True)
        new_state.step += 1
        if action.kind == 'tool_call':
            new_state.tool_results.append(observation['output'])
        return new_state

### The initial state

State is where the agent is, not where the conversation is. It carries the task, the step counter, the accumulated tool results and a status the loop reads to decide whether to continue. The task names the goal and the inputs the agent acts on.

In [ ]:
from forgeloop.agents.core.state import AgentState
from forgeloop.agents.core.task import TaskSpec

task = TaskSpec(
    goal='classify a customer complaint message',
    inputs={'message': 'I was charged a $35 overdraft fee I did not authorize.'},
    expected_outputs=['category'],
    constraints=[],
    validation=[],
)
state = AgentState(
    task=task, step=0, messages=[], scratchpad_entries=[],
    tool_results=[], status='running', final_output=None,
)

### Running the loop

`run_loop` is a generator: it yields a `StepRecord` for every step, exposing what the agent proposed, what the environment returned and how the state changed. The loop is inspectable by construction, which is what later chapters build evaluation and governance on top of.

In [ ]:
from forgeloop.agents.core.loop import run_loop

agent = ClassifyThenFinish()
env = ToolEnvironment(registry)

for record in run_loop(agent, env, state, max_steps=4):
    print(f'step {record.step}: {record.action.kind:10s} '
          f'-> status {record.state_after.status}')
    if record.action.kind == 'tool_call':
        print('   observation:', record.observation['output'])
    if record.action.kind == 'finish':
        print('   final output:', record.action.output)

The agent now acts: it proposes a typed call, the environment runs it, and the loop records the transition to a finished state. This is the substrate the rest of the capstone attaches to. Chapter~3 records a reasoning trace alongside each step, Chapter~5 widens the registry to the full five-tool action space, and Chapter~6 inserts the governance gates between the proposal and the environment call so that an unsafe action is refused before it runs.